# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- 1. Load the data ---
# Load the dataset from the provided CSV file
df = pd.read_csv('/content/hotels (1).csv')

print(f"Dataset initially has {df.shape[0]} rows and {df.shape[1]} columns.")
print("\nFirst 5 rows of the dataset:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nMissing values before cleaning:")
display(df.isnull().sum()[df.isnull().sum() > 0])

# --- 2. Data Cleaning and Feature Engineering ---

# Drop columns that leak information about cancellation ('reservation_status' and 'reservation_status_date')
df = df.drop(['reservation_status', 'reservation_status_date'], axis=1)

# Handle missing values
# For 'country', 'agent', 'company', fill NaN with 'Unknown' as they are categorical/ID-like
df['country'].fillna('Unknown', inplace=True)
df['agent'].fillna('Unknown', inplace=True)
df['company'].fillna('Unknown', inplace=True)

# For 'children', fill NaN with 0 (most common sense imputation for count of children)
df['children'].fillna(0, inplace=True)

# Correct data types for 'children', 'agent', 'company'
df['children'] = df['children'].astype(int)
df['agent'] = df['agent'].astype(str)
df['company'] = df['company'].astype(str)

# Clean 'meal' column: 'Undefined' and 'SC' refer to the same thing (no meal package)
df['meal'] = df['meal'].replace(['Undefined', 'SC'], 'No Meal')

# Feature Engineering: Create 'total_nights' and 'total_guests'
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']

# Drop original columns if combined into new ones and no longer needed
df = df.drop(['stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies'], axis=1)

# Remove rows with illogical booking characteristics (no guests or no nights)
df = df[df['total_guests'] > 0]
df = df[df['total_nights'] > 0]

# Handle ADR (Average Daily Rate): Replace negative values with 0, then 0 with the median ADR
df.loc[df['adr'] < 0, 'adr'] = 0
median_adr = df['adr'].median()
df.loc[df['adr'] == 0, 'adr'] = median_adr

print("\nMissing values after cleaning:")
display(df.isnull().sum()[df.isnull().sum() > 0]) # Should be empty
print(f"\nDataset has {df.shape[0]} rows and {df.shape[1]} columns after cleaning and feature engineering (before X,y split).")

# --- 3. Create X (features) and y (target) and Split the data ---
# Define target variable (y) and features (X)
y = df['is_canceled']
X = df.drop('is_canceled', axis=1)

# Split data into training and test sets (70/30) with a fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# --- 4. Encode Categorical Variables (after split to prevent data leakage) ---

# Identify categorical and numerical features from X_train
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns
numerical_features = X_train.select_dtypes(include=np.number).columns # Use np.number to include all numerical types

# Create a column transformer for one-hot encoding categorical features
# 'passthrough' for numerical features means they will be kept as is
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    remainder='passthrough' # Keep any other columns that are not explicitly handled
)

# Apply the preprocessing to training and test data
# We fit the preprocessor on X_train only to avoid data leakage from the test set
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nX_train_processed shape after encoding: {X_train_processed.shape}")
print(f"X_test_processed shape after encoding: {X_test_processed.shape}")

print("\nTypes of features identified for encoding:")
print(f"Categorical features: {list(categorical_features)}")
print(f"Numerical features: {list(numerical_features)}")


Dataset initially has 9404 rows and 32 columns.

First 5 rows of the dataset:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9404 entries, 0 to 9403
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           9404 non-null   object 
 1   is_canceled                     9404 non-null   int64  
 2   lead_time                       9404 non-null   int64  
 3   arrival_date_year               9404 non-null   int64  
 4   arrival_date_month              9404 non-null   object 
 5   arrival_date_week_number        9404 non-null   int64  
 6   arrival_date_day_of_month       9404 non-null   int64  
 7   stays_in_weekend_nights         9404 non-null   int64  
 8   stays_in_week_nights            9404 non-null   int64  
 9   adults                          9404 non-null   int64  
 10  children                        9404 non-null   int64  
 11  babies                          9404 non-null   int64  
 12  meal               

,0
country,266
agent,2335
company,8393



Missing values after cleaning:


/tmp/ipykernel_23960/637184354.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna('Unknown', inplace=True)
/tmp/ipykernel_23960/637184354.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

,0



Dataset has 9283 rows and 27 columns after cleaning and feature engineering (before X,y split).

X_train shape: (6498, 26)
X_test shape: (2785, 26)
y_train shape: (6498,)
y_test shape: (2785,)

X_train_processed shape after encoding: (6498, 330)
X_test_processed shape after encoding: (2785, 330)

Types of features identified for encoding:
Categorical features: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent', 'company', 'customer_type']
Numerical features: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests']


### ✍️ Your Response: 🔧
1. The dataset has 9283 rows and 27 rows after cleanign and feature engineering.

2. Categorial feature has 12 features, and numerical has 14 features.
3. I used handling missing values by dropping, corrected the data type, cleaning categorial value, and engineeting features.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# --- 1. Train a Naïve Bayes classifier ---
# Initialize the Gaussian Naïve Bayes model
naive_bayes_model = GaussianNB()

# Train the model on the processed training data
naive_bayes_model.fit(X_train_processed.toarray(), y_train) # .toarray() is often needed for sparse matrices from OneHotEncoder

# --- 2. Use it to predict on your test data ---
y_pred_nb = naive_bayes_model.predict(X_test_processed.toarray())

# --- 3. Print a classification report and confusion matrix ---
print("\n--- Naïve Bayes Model Performance ---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))



--- Naïve Bayes Model Performance ---

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.46      0.62      2082
           1       0.38      0.97      0.54       703

    accuracy                           0.59      2785
   macro avg       0.68      0.71      0.58      2785
weighted avg       0.83      0.59      0.60      2785


Confusion Matrix:
[[ 954 1128]
 [  21  682]]


### ✍️ Your Response: 🔧
1. This data show the 59% of accuracy, while recall of canceled percentage show 97%, precision of cancel show 38%, and f1 score which show accuracy of cancel of class one show 54%.
2. In this case, we can predict that we can use that for managing overbooking and in-stock rooms, react from customers.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# --- 1. Scale the data ---
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit on X_train_processed and transform both X_train_processed and X_test_processed
# Convert sparse matrix to dense array for scaling and SVM
X_train_scaled = scaler.fit_transform(X_train_processed.toarray())
X_test_scaled = scaler.transform(X_test_processed.toarray())

print("Data scaled successfully.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

# --- 2. Train an SVM classifier (use linear kernel) ---
# Initialize the SVC model with a linear kernel
# Set random_state for reproducibility
svm_model = SVC(kernel='linear', random_state=42)

print("\nTraining SVM model... This may take several minutes.")
svm_model.fit(X_train_scaled, y_train)

print("SVM model trained.")

# --- 3. Make predictions and evaluate with classification metrics ---
y_pred_svm = svm_model.predict(X_test_scaled)

print("\n--- Support Vector Machine Model Performance ---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))


Data scaled successfully.
X_train_scaled shape: (6498, 330)
X_test_scaled shape: (2785, 330)

Training SVM model... This may take several minutes.
SVM model trained.

--- Support Vector Machine Model Performance ---

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      2082
           1       0.80      0.83      0.82       703

    accuracy                           0.91      2785
   macro avg       0.87      0.88      0.88      2785
weighted avg       0.91      0.91      0.91      2785


Confusion Matrix:
[[1938  144]
 [ 119  584]]


### ✍️ Your Response:
1. I used support vactor machine, which show 91% accuracy. In this case, the data shows that precrision of class 1 is 80%, recall of cancel of class 1 is 83%, F1 score is 82%, which mean that they had stable.
2. Support vactor machine would be the best option for complex relationship between features and target variables,and classifying for clarifying for specific purpose. In that situation, it is good for classifying for clariying maximum margin.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [5]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split as split_for_validation # Renaming to avoid confusion with X_train, X_test

# --- 1. Build a MLPClassifier model ---
# The X_train_scaled and y_train are already prepared from the SVM step.
# We need to create a validation set from the training data.

# Split X_train_scaled and y_train into training and validation sets
X_train_nn, X_val_nn, y_train_nn, y_val_nn = split_for_validation(X_train_scaled, y_train, test_size=0.2, random_state=42)

print(f"Neural Network training data shape: {X_train_nn.shape}")
print(f"Neural Network validation data shape: {X_val_nn.shape}")

# Initialize MLPClassifier with a simple architecture (e.g., 2 hidden layers)
# using 'relu' activation and 'adam' solver which are good defaults.
# max_iter is set to a reasonable number, and early_stopping can be used.
# random_state for reproducibility.
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50), # Two hidden layers with 100 and 50 neurons respectively
    activation='relu',
    solver='adam',
    max_iter=300, # Increased max_iter for better convergence
    random_state=42,
    early_stopping=True, # Stop training when validation score does not improve
    validation_fraction=0.1, # Fraction of training data to set aside for early stopping validation
    n_iter_no_change=10, # Number of iterations with no improvement to wait before stopping
    verbose=True # To see the training progress
)

print("\nTraining Neural Network model... This may take some time.")
mlp_model.fit(X_train_nn, y_train_nn)

print("Neural Network model trained.")

# --- 2. Evaluate accuracy and performance on the test data ---
y_pred_nn = mlp_model.predict(X_test_scaled)

print("\n--- Neural Network Model Performance ---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))


Neural Network training data shape: (5198, 330)
Neural Network validation data shape: (1300, 330)

Training Neural Network model... This may take some time.
Iteration 1, loss = 0.50879468
Validation score: 0.819231
Iteration 2, loss = 0.33478707
Validation score: 0.880769
Iteration 3, loss = 0.25216044
Validation score: 0.896154
Iteration 4, loss = 0.20886535
Validation score: 0.900000
Iteration 5, loss = 0.18941965
Validation score: 0.900000
Iteration 6, loss = 0.17451759
Validation score: 0.905769
Iteration 7, loss = 0.16426510
Validation score: 0.901923
Iteration 8, loss = 0.15723678
Validation score: 0.905769
Iteration 9, loss = 0.14985837
Validation score: 0.913462
Iteration 10, loss = 0.14244504
Validation score: 0.911538
Iteration 11, loss = 0.13730871
Validation score: 0.911538
Iteration 12, loss = 0.13265227
Validation score: 0.909615
Iteration 13, loss = 0.12653101
Validation score: 0.905769
Iteration 14, loss = 0.12220173
Validation score: 0.907692
Iteration 15, loss = 0.119

### ✍️ Your Response: 🔧
1. Neural network model show the most stable and highest score in both accuracy of all part than others, sine it show 81% of precision, 83%% of recall, 82% of f1 score and 91% accuracy.

2. Since neural network model has the most powerful performance of prediction, it would be good to use. However, since it is hard to see the insight (details of data), we should think deeper about it.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [6]:
from sklearn.metrics import classification_report
import pandas as pd

# --- Collect performance metrics ---

# Naïve Bayes metrics
report_nb = classification_report(y_test, y_pred_nb, output_dict=True)
accuracy_nb = report_nb['accuracy']
f1_canceled_nb = report_nb['1']['f1-score']
precision_canceled_nb = report_nb['1']['precision']
recall_canceled_nb = report_nb['1']['recall']

# SVM metrics
report_svm = classification_report(y_test, y_pred_svm, output_dict=True)
accuracy_svm = report_svm['accuracy']
f1_canceled_svm = report_svm['1']['f1-score']
precision_canceled_svm = report_svm['1']['precision']
recall_canceled_svm = report_svm['1']['recall']

# Neural Network metrics
report_nn = classification_report(y_test, y_pred_nn, output_dict=True)
accuracy_nn = report_nn['accuracy']
f1_canceled_nn = report_nn['1']['f1-score']
precision_canceled_nn = report_nn['1']['precision']
recall_canceled_nn = report_nn['1']['recall']

# --- Create a DataFrame for easy comparison ---
comparison_df = pd.DataFrame({
    'Model': ['Naïve Bayes', 'SVM', 'Neural Network'],
    'Overall Accuracy': [accuracy_nb, accuracy_svm, accuracy_nn],
    'F1-Score (Canceled)': [f1_canceled_nb, f1_canceled_svm, f1_canceled_nn],
    'Precision (Canceled)': [precision_canceled_nb, precision_canceled_svm, precision_canceled_nn],
    'Recall (Canceled)': [recall_canceled_nb, recall_canceled_svm, recall_canceled_nn]
})

print("\n--- Model Performance Comparison ---")
display(comparison_df.sort_values(by='F1-Score (Canceled)', ascending=False))

# --- Summarize which model performed best ---
print("\n--- Summary of Best Model ---")
best_model_f1 = comparison_df.loc[comparison_df['F1-Score (Canceled)'].idxmax()]
print(f"The model with the best F1-Score for predicting cancellations is the {best_model_f1['Model']} model, with an F1-Score of {best_model_f1['F1-Score (Canceled)']:.2f}.")
print(f"It also achieved the highest overall accuracy of {best_model_f1['Overall Accuracy']:.2f}.")



--- Model Performance Comparison ---


,Model,Overall Accuracy,F1-Score (Canceled),Precision (Canceled),Recall (Canceled)
2,Neural Network,0.906643,0.816901,0.808926,0.825036
1,SVM,0.905566,0.816212,0.802198,0.830725
0,Naïve Bayes,0.587433,0.542778,0.376796,0.970128



--- Summary of Best Model ---
The model with the best F1-Score for predicting cancellations is the Neural Network model, with an F1-Score of 0.82.
It also achieved the highest overall accuracy of 0.91.


### ✍️ Your Response: 🔧
1. Neural Network Model is the best model to use since it had highest overall accuracy.

2. Yes, since all accuracy of each part is high, and it is the most stable model than any others.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend to use Neural Network for implementing since it is the most stable way with high accuracy. It would work well in resource optimization and marketing strategy. However, it has limitation to see insights of data analysis, and still ahve overfitting since accuracy is not 100%. Also, I hopely it works at the customer loyalty analysis like how many cancellation of customers happened and how to reduce it.
2. As thinking to work at logistics industry, there would be similar problem like hotel examples, since the food and beverage problem is related to logistics, and also logistics has problem of delivery cancellation. It would be work if I implement this.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"